# Step 1 – Base heating network (DHNx)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Supply **5 consumers** from **2 heat producers** using **2 pipe types (DN-30, DN-50)**. DHNx decides which candidate pipes are built and which DN each one gets. |
| **Reads** | `data/Demand profiles_heating.xlsx` |
| **Writes** | `data/step1/` (DHNx input CSVs) · `results/step1/` (pipe results, summary, figures) |
| **Grading** | E – build/run the base network, verify both pipe types are used, write down the design problem, justify all inputs |
| **Tutorial reference** | `DHNx Tutorial/DHNx.ipynb` → *introduction* and *discrete_DN_numbers* |

**How to use:** run all cells from top to bottom. The very first time on a new computer, section 0 creates the Python environment.

**Legend used in all notebooks**
- 🔧 **ASSUMPTION** – a value you must state, give units for, and justify (citation or engineering reasoning) in the report.
- ✏️ **TODO** – a group decision or an answer you must write.

## 0. Setup (run once per computer)

### 0.1 Environment setup cell
**What this cell does**
1. Checks whether all required packages are importable in the *current* kernel → if yes, prints ✅ and does nothing else.
2. If not, it creates a virtual environment `.venv/` inside this repo, installs `requirements.txt` into it and registers it as a Jupyter kernel called **"Python (MJ2505 DH project)"**.
3. It then stops and asks you to **switch kernel** (VS Code: *Select Kernel → Python Environments → .venv*; JupyterLab: *Kernel → Change kernel*) and run the notebook again. From then on the cell just prints ✅.

**Solver:** the course uses **CBC** (`conda install -c conda-forge coin-or-cbc`). If CBC is installed it is used automatically; otherwise the pip-installable **HiGHS** solver is used, so the notebooks run without conda. Both give the same optimum.

Needs Python **3.11–3.14** (the course uses 3.12). If the kernel you opened runs another version, the cell looks for an installed Python 3.11–3.14 to build `.venv` with.

In [ ]:
# 0.1 ENVIRONMENT SETUP - safe to re-run, does nothing if everything is already installed
import sys, subprocess, importlib.util, platform, shutil
from pathlib import Path

REPO = Path.cwd()                      # Jupyter starts in the notebook's folder = repo root
VENV = REPO / ".venv"
KERNEL_NAME, KERNEL_DISPLAY = "mj2505-dh", "Python (MJ2505 DH project)"
REQUIRED = ["dhnx", "oemof.solph", "pyomo", "highspy", "pandas", "numpy", "matplotlib", "openpyxl", "ipykernel"]
SUPPORTED = [(3, 12), (3, 13), (3, 11), (3, 14)]      # preference order


def is_installed(pkg):
    try:
        return importlib.util.find_spec(pkg) is not None
    except ModuleNotFoundError:
        return False


def python_version(cmd):
    try:
        out = subprocess.run(cmd + ["-c", "import sys; print(sys.version_info[0], sys.version_info[1])"],
                             capture_output=True, text=True, timeout=20)
        return tuple(int(x) for x in out.stdout.split()) if out.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired, ValueError):
        return None


def find_base_python():
    # this kernel's Python if supported, otherwise another installed Python 3.11-3.14
    if sys.version_info[:2] in SUPPORTED:
        return [sys.executable]
    candidates = [["py", f"-{a}.{b}"] for a, b in SUPPORTED] if platform.system() == "Windows" else \
                 [[exe] for a, b in SUPPORTED if (exe := shutil.which(f"python{a}.{b}"))]
    for cmd in candidates:
        if python_version(cmd) in SUPPORTED:
            return cmd
    raise RuntimeError("No Python 3.11-3.14 found. Install Python 3.12 from python.org and re-run this cell.")


missing = [p for p in REQUIRED if not is_installed(p)]
venv_python = VENV / ("Scripts/python.exe" if platform.system() == "Windows" else "bin/python")
running_in_project_venv = Path(sys.prefix).resolve() == VENV.resolve()

if not missing:
    print(f"✅ Environment OK - Python {sys.version.split()[0]} ({sys.executable})")

elif running_in_project_venv:
    print("Installing missing packages into the project venv:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
    print("✅ Done - continue running the notebook.")

else:
    if not venv_python.exists():
        base = find_base_python()
        print("Creating virtual environment", VENV, "with", " ".join(base))
        subprocess.check_call(base + ["-m", "venv", str(VENV)])
    print("Installing requirements.txt into .venv (a few minutes the first time) ...")
    subprocess.check_call([str(venv_python), "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([str(venv_python), "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
    subprocess.check_call([str(venv_python), "-m", "ipykernel", "install", "--user",
                           "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY])
    raise SystemExit(f"\n✅ Environment created.\n👉 Now switch the notebook kernel to '{KERNEL_DISPLAY}' "
                     "(or the '.venv' interpreter), restart it and run the notebook again from the top.")

### 0.2 Imports and diagnostic
Same libraries as the DHNx tutorial. `helpers.py` is the project's only supporting code file (paths, solver choice, small helper functions shared by all steps).
The messages *"Need to install geopandas/shapely/osmnx/CoolProp ..."* printed by DHNx are harmless (only needed for OpenStreetMap import / hydraulics, which we do not use).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import dhnx
import pandas as pd
import numpy as np
import time

from helpers import (DATA, RESULTS, SOLVER, SOLVER_OPTIONS, COUNTRY, T_SUPPLY, T_RETURN, DELTA_T, INTEREST_RATE,
                     PIPE_LIFETIME, epc, load_demand_profiles, pipe_capacity_kW, write_dhnx_input,
                     plot_network)

OUT = RESULTS / "step1"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Project-wide assumptions
These values are defined **once** in `helpers.py` (section 0) and reused by every step, so the assumptions stay consistent as the project description requires.
Change them there – never redefine them inside a notebook.

| Parameter | Value | Unit | 🔧 Justification (fill in) |
|---|---|---|---|
| Country | Sweden | – | ✏️ chosen country – needed in Step 3 for prices, grid emission factor and solar profile |
| Supply / return temperature | 80 / 40 | °C | ✏️ cite typical Swedish DH temperatures |
| Interest rate | 5 | % | ✏️ same as tutorial `epc()` default – justify |
| Pipe lifetime | 40 | years | ✏️ cite lifetime of pre-insulated pipes |

In [ ]:
pd.Series({"Country": COUNTRY, "Supply temperature [°C]": T_SUPPLY, "Return temperature [°C]": T_RETURN,
           "ΔT [K]": DELTA_T, "Interest rate [-]": INTEREST_RATE, "Pipe lifetime [a]": PIPE_LIFETIME},
          name="value").to_frame()

## 2. Heat demand – choose five consumers

### 2.1 Load the 50 demand profiles
`load_demand_profiles()` reads *Sheet1* of the Excel file (8760 hourly values × 50 profiles). The first run caches it as `data/demand_profiles.csv` (the Excel takes ~30 s to read).

🔧 **ASSUMPTION – units:** the values are interpreted as **kW** (hourly mean heat load, so 1 value = kWh in that hour). Justify from the magnitudes (peak ≈ 50 kW – 3.8 MW → single houses up to large buildings).

The table shows peak, mean, annual energy and full-load hours (annual energy / peak) for each profile, to help you choose.

In [ ]:
demand = load_demand_profiles()                     # kW, index = hour 0..8759

profile_stats = pd.DataFrame({
    "peak [kW]": demand.max(),
    "mean [kW]": demand.mean(),
    "annual [MWh]": demand.sum() / 1000,
    "full-load hours [h]": demand.sum() / demand.max(),
}).round(1)
profile_stats.sort_values("peak [kW]")

### 2.2 Select the five consumers
✏️ **TODO – choose your own five profiles and justify the choice** (e.g. a mix of building types/sizes, seasonal vs. flat profiles).
The choice is coupled to the pipe capacities in section 3: a consumer whose peak exceeds the DN-30 capacity can only be connected with DN-50, and the sum of the peaks must fit through the pipes leaving the producers.

The placeholder below uses small/medium profiles so that both DN-30 and DN-50 are needed.
`P_heat_max` (peak of each profile) is what DHNx uses for the design, see section 6.

In [ ]:
# 🔧 ASSUMPTION - selected demand profiles (placeholder, change and justify)
STEP1_LOADS = ["Load 2", "Load 4", "Load 10", "Load 5", "Load 6"]   # consumer 0..4

print(profile_stats.loc[STEP1_LOADS])
print(f"\nSum of individual peaks : {demand[STEP1_LOADS].max().sum():8.1f} kW")
print(f"Peak of the summed load : {demand[STEP1_LOADS].sum(axis=1).max():8.1f} kW  (simultaneity factor "
      f"{demand[STEP1_LOADS].sum(axis=1).max() / demand[STEP1_LOADS].max().sum():.2f})")

fig, ax = plt.subplots(figsize=(14, 4))
demand[STEP1_LOADS].plot(ax=ax, lw=0.6)
ax.set_xlabel("Hour of the year"); ax.set_ylabel("Heat demand [kW]"); ax.set_title("Selected consumer profiles (Step 1)")
plt.tight_layout(); plt.savefig(OUT / "demand_profiles.png", dpi=150); plt.show()

## 3. Pipe types DN-30 and DN-50

DHNx reads the pipe types from `invest_data/network/pipes.csv` – one row per DN (same format as the tutorial's *discrete_DN_numbers* example):

| DHNx column | Meaning | Unit | How we set it |
|---|---|---|---|
| `label_3` | pipe type name | – | DN-30, DN-50 |
| `nonconvex` | 1 = built-or-not binary decision with fixed cost/loss (MILP) | – | 1 |
| `cap_max` / `cap_min` | max / min thermal capacity of the pipe | kW | from velocity limit, see below / 0 |
| `l_factor_fix` | heat loss of a built pipe, independent of flow | kW/m | 🔧 manufacturer data (W/m ÷ 1000) |
| `l_factor` | heat loss proportional to capacity | 1/m | 0 (losses depend on temperature, not flow) |
| `fix_costs` | cost of building the pipe | €/(m·a) | 🔧 investment €/m, annualised with `epc()` |
| `capex_pipes` | cost per kW of capacity | €/(kW·m·a) | 1e-4 (negligible – only makes DHNx report the *used* capacity instead of an arbitrary value) |

**Capacity (engineering reasoning):**  $\dot Q_{max} = \rho \, v_{max} \, \frac{\pi d_i^2}{4} \, c_p \, \Delta T$, with $d_i \approx$ DN [mm], $\Delta T$ = supply − return, and a maximum velocity that keeps the pressure drop reasonable (larger pipes tolerate higher velocity).

🔧 **ASSUMPTIONS to justify:** `v_max`, investment cost €/m, heat loss W/m per DN (e.g. pipe manufacturer catalogue for twin pre-insulated pipes; Danish Energy Agency technology data; Swedish cost reports). The tutorial's values (466–620 €/m, 7.7–11.4 W/m) are a good sanity check.

In [ ]:
# 🔧 ASSUMPTIONS - pipe catalogue (placeholders, replace with cited values)
pipe_catalog = pd.DataFrame({
    "label_3":          ["DN-30", "DN-50"],
    "dn_mm":            [30,      50],       # inner diameter ≈ DN [mm]
    "v_max":            [1.1,     1.5],      # m/s   max flow velocity
    "invest_eur_per_m": [470,     560],      # €/m   trench + twin pipe, installed
    "loss_W_per_m":     [8.3,     9.8],      # W/m   heat loss at design temperatures
})
pipe_catalog["cap_max_kW"] = pipe_capacity_kW(pipe_catalog["dn_mm"], pipe_catalog["v_max"], DELTA_T).round(1)
pipe_catalog["annualised_eur_per_m_a"] = epc(pipe_catalog["invest_eur_per_m"], i=INTEREST_RATE, n=PIPE_LIFETIME).round(2)

# -> DHNx format (invest_data/network/pipes.csv)
pipe_types = pd.DataFrame({
    "label_3":      pipe_catalog["label_3"],
    "active":       1,
    "nonconvex":    1,
    "l_factor":     0,
    "l_factor_fix": pipe_catalog["loss_W_per_m"] / 1000,      # kW/m
    "cap_max":      pipe_catalog["cap_max_kW"],
    "cap_min":      0,
    "capex_pipes":  1e-4,
    "fix_costs":    pipe_catalog["annualised_eur_per_m_a"],
})
display(pipe_catalog)
display(pipe_types)

## 4. Network layout – producers, consumers, forks and candidate pipes

Like the tutorial `twn_data` folder we define four tables. Coordinates are in **metres** on a local grid (`lon` = x, `lat` = y); pipe lengths are computed from the coordinates by `write_dhnx_input()`.

- **consumers** – id, position, `P_heat_max` [kW] (peak of the chosen profile) and `profile` (which Excel column it is – used again in Step 2–3).
- **producers** – id, position, plus two *producer-specific* DHNx attributes:
  - `heat.source.nominal_capacity` [kW] – 🔧 max heat output of the plant at the design hour.
  - `heat.source.variable_costs` – 🔧 operating cost. DHNx optimises a single design hour, so to make operating cost comparable with the *annualised* pipe cost we enter `cost [€/kWh] × full-load hours of the demand [h/a]` → €/(kW·a). This approximation must be stated in the report.
- **forks** – junctions of the street network.
- **pipes** – the *candidate* connections (superstructure). Include alternative routes/loops: DHNx chooses which ones to build.

Rules from DHNx: consumers can only be connected **from a fork**, producers only **to a fork**, and the graph must be connected.

✏️ **TODO:** design your own layout and producer types. Placeholder: P0 = biomass boiler (cheap, base load), P1 = natural-gas boiler (expensive, peak). Keep producer ids – Step 2 retains them.

In [ ]:
# 🔧 ASSUMPTIONS - layout (placeholder)
consumers = pd.DataFrame({
    "id":         [0,    1,    2,   3,   4],
    "lon":        [100,  200,  300, 160, 240],          # x [m]
    "lat":        [-60,  -60,  -60, 130, 130],          # y [m]
    "profile":    STEP1_LOADS,
})
consumers["P_heat_max"] = demand[STEP1_LOADS].max().round(1).values   # kW

FLH = demand[STEP1_LOADS].sum().sum() / demand[STEP1_LOADS].sum(axis=1).max()   # full-load hours [h/a]

producers = pd.DataFrame({
    "id":         [0, 1],
    "lon":        [0, 400],
    "lat":        [0, 0],
    "active":     1,
    "technology": ["biomass boiler", "gas boiler"],
    # 🔧 rough marginal heat cost [€/MWh heat] = fuel price / efficiency + O&M. Keep consistent with Step 3 data!
    "op_cost_eur_per_MWh": [40, 60],
    "heat.source.nominal_capacity": [600, 600],          # kW, max output at the design hour
})
producers["heat.source.variable_costs"] = producers["op_cost_eur_per_MWh"] / 1000 * FLH   # €/(kW·a)

forks = pd.DataFrame({
    "id":  [0,   1,   2,   3,   4],
    "lon": [100, 200, 300, 100, 300],
    "lat": [0,   0,   0,   100, 100],
})

# candidate pipes (from_node -> to_node); DHNx decides which ones are built
edges = [
    ("producers-0", "forks-0"), ("producers-0", "forks-3"), ("producers-1", "forks-2"),   # plant connections
    ("forks-0", "forks-1"), ("forks-1", "forks-2"),                                        # main street
    ("forks-0", "forks-3"), ("forks-2", "forks-4"), ("forks-3", "forks-4"),                # side streets (loop)
    ("forks-0", "consumers-0"), ("forks-1", "consumers-1"), ("forks-2", "consumers-2"),    # house connections
    ("forks-3", "consumers-3"), ("forks-4", "consumers-4"),
]
pipes = pd.DataFrame({"id": range(len(edges)),
                      "from_node": [a for a, _ in edges],
                      "to_node":   [b for _, b in edges]})

print(f"Full-load hours of the Step 1 demand: {FLH:.0f} h/a")
twn_data_path, invest_data_path = write_dhnx_input(DATA / "step1", consumers, producers, forks, pipes, pipe_types)
print("DHNx input written to", twn_data_path.parent)

## 5. Load the network into DHNx and plot the given (candidate) network
Exactly the tutorial workflow: `ThermalNetwork()` → `from_csv_folder(twn_data)` → `load_invest_options(invest_data)`.

In [ ]:
# Initialize thermal network
network = dhnx.network.ThermalNetwork()

# Load town parameter
network = network.from_csv_folder(str(twn_data_path))

# Load investment parameter
invest_opt = dhnx.input_output.load_invest_options(str(invest_data_path))

# plot initial/given network
plot_network(network, "Given network (all candidate pipes) – Step 1")
plt.savefig(OUT / "network_given.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. The network design problem (write this down in the report)

DHNx builds an oemof.solph model of the network and solves a **mixed-integer linear program (MILP)** for one design time step (`heat_demand='scalar'` → every consumer at its peak `P_heat_max`).

**Decision variables**
- $x_{e,k} \in \{0,1\}$ – build pipe type $k$ (DN-30/DN-50) on candidate edge $e$ (per flow direction)
- $P_{e,k} \ge 0$ [kW] – capacity / heat flow of that pipe
- $Q_p \ge 0$ [kW] – heat supplied by producer $p$

**Objective** (annual cost, €/a)
$$\min \; \sum_e \sum_k L_e\,\big(c^{fix}_k\,x_{e,k} + c^{cap}_k\,P_{e,k}\big) \; + \; \sum_p c_p\,Q_p$$

**Constraints**
- heat balance at every fork and consumer: inflow − outflow − losses = demand ($P^{heat}_{max}$ at consumers, 0 at forks)
- pipe capacity: $P_{e,k} \le \overline{P}_k \, x_{e,k}$
- pipe losses: $\dot Q^{loss}_{e} = L_e \, l^{fix}_k \, x_{e,k}$ (must also be supplied by the producers)
- producer capacity: $Q_p \le \overline{Q}_p$
- at most one pipe type per edge and direction

✏️ **TODO:** explain in words how cost, capacity and heat loss drive the pipe choice (see questions at the end).

### 6.1 Run the optimisation
Tutorial call: `network.optimize_investment(invest_options=invest_opt, solver='cbc')`. We pass `solver=SOLVER` (CBC if installed, else HiGHS) and hide the solver log (`tee=False` and `SOLVER_OPTIONS`; set `tee=True` to see it with CBC).

In [ ]:
t0 = time.perf_counter()
network.optimize_investment(invest_options=invest_opt, solver=SOLVER,
                             solve_kw={"tee": False}, solver_cmdline_options=SOLVER_OPTIONS)
solve_time = time.perf_counter() - t0
print(f"Solved in {solve_time:.1f} s")

## 7. Results

### 7.1 Selected pipes (tutorial post-processing)
`network.results.optimization` is a dictionary with `oemof` (energy flows), `oemof_meta` (objective, solver info) and `components` (the pipe results).
For each candidate pipe: chosen type `hp_type`, `capacity` [kW] (= heat it must carry), `direction`, `costs` [€/a] and `losses` [kW].

In [ ]:
# get results
results_edges = network.results.optimization["components"]["pipes"]
print(results_edges[["from_node", "to_node", "length", "hp_type", "capacity", "direction", "costs", "losses"]])
print("\nObjective value: ", network.results.optimization["oemof_meta"]["objective"])

built = results_edges[results_edges["capacity"] > 0.001].copy()
built = built.merge(pipe_catalog[["label_3", "cap_max_kW", "invest_eur_per_m"]],
                    left_on="hp_type", right_on="label_3", how="left").drop(columns="label_3")
built["utilisation [%]"] = (100 * built["capacity"] / built["cap_max_kW"]).round(1)

# Requirement of the project description: BOTH pipe types must be used
used_types = sorted(built["hp_type"].unique())
print("\nPipe types used:", used_types)
assert set(used_types) == set(pipe_catalog["label_3"]), "✏️ Not all pipe types are used - adjust layout/loads/capacities!"
print("✅ Both pipe types are used")

### 7.2 Plot the optimised network
Tutorial code: keep only pipes with `capacity > 0.001` and redraw; DN labels as in the *discrete_DN_numbers* example.

In [ ]:
# assign new ThermalNetwork with invested pipes
twn_results = network
twn_results.components["pipes"] = results_edges[results_edges["capacity"] > 0.001]

plot_network(twn_results, "Result network – Step 1", show_dn=True)
plt.savefig(OUT / "network_result.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Producer output and key numbers
Heat supplied by each producer at the design hour (from the oemof results inside DHNx) and a summary that Step 2 compares against.
- **Annual heat loss** = design-hour loss [kW] × 8760 h – valid because losses are flow-independent (`l_factor = 0`).
- **Pipe investment** = Σ length × €/m (not annualised); **annual pipe cost** = Σ DHNx `costs`.
- **Producer operating cost** = objective − annual pipe cost.

In [ ]:
oemof_results = network.results.optimization["oemof"]
producer_supply = pd.Series({
    "P" + str(k[0]).split("producers-")[-1]: v["sequences"]["flow"].iloc[0]
    for k, v in oemof_results.items() if "source_producers" in str(k[0])
}, name="heat at design hour [kW]")
producer_table = producer_supply.to_frame()
producer_table["capacity [kW]"] = producers.set_index("id")["heat.source.nominal_capacity"].values
producer_table["utilisation [%]"] = (100 * producer_table.iloc[:, 0] / producer_table["capacity [kW]"]).round(1)
producer_table["technology"] = producers["technology"].values
display(producer_table)

objective = network.results.optimization["oemof_meta"]["objective"]
summary = pd.Series({
    "consumers": len(consumers),
    "producers": len(producers),
    "pipe types available": len(pipe_types),
    "candidate pipes": len(pipes),
    "built pipes": len(built),
    "built length [m]": built["length"].sum(),
    "pipe investment [EUR]": (built["length"] * built["invest_eur_per_m"]).sum(),
    "annual pipe cost [EUR/a]": built["costs"].sum(),
    "producer operating cost [EUR/a]": objective - built["costs"].sum(),
    "objective [EUR/a]": objective,
    "design demand [kW]": consumers["P_heat_max"].sum(),
    "heat losses [kW]": built["losses"].sum(),
    "heat losses [MWh/a]": built["losses"].sum() * 8760 / 1000,
    "annual demand [MWh/a]": demand[STEP1_LOADS].sum().sum() / 1000,
    "solve time [s]": solve_time,
})
summary["losses / demand [%]"] = 100 * summary["heat losses [MWh/a]"] / summary["annual demand [MWh/a]"]
summary.round(2).to_frame("Step 1")

## 8. Compare the selected pipe types (cost, thermal losses, capacity)
Per type: how many metres were built, cost and losses per metre, capacity, and the specific cost/loss per kW of capacity – the numbers you need to explain *why* each type is chosen where it is.

In [ ]:
pipe_comparison = built.groupby("hp_type").agg(
    pipes=("length", "size"),
    length_m=("length", "sum"),
    annual_cost_eur=("costs", "sum"),
    losses_kW=("losses", "sum"),
    max_utilisation_pct=("utilisation [%]", "max"),
).join(pipe_catalog.set_index("label_3")[["invest_eur_per_m", "loss_W_per_m", "cap_max_kW"]])
pipe_comparison["EUR per m per kW capacity"] = (pipe_comparison["invest_eur_per_m"] / pipe_comparison["cap_max_kW"]).round(2)
pipe_comparison["W loss per m per kW capacity"] = (pipe_comparison["loss_W_per_m"] / pipe_comparison["cap_max_kW"]).round(4)
pipe_comparison

## 9. Save results (used by Step 2)

In [ ]:
built.to_csv(OUT / "pipes_results.csv", index=False)
producer_table.to_csv(OUT / "producers_results.csv")
pipe_comparison.to_csv(OUT / "pipe_type_comparison.csv")
summary.to_csv(OUT / "summary.csv", header=["Step 1"])
print("Saved to", OUT)

## 10. Questions to answer (report / presentation)
✏️ Write your answers here, supported by the numbers above.

1. **How does the optimizer decide which pipe type to use in different parts of the network?**
   *Hint: MILP – for each edge it needs a type whose `cap_max` ≥ the heat that must pass; among feasible types it takes the one with the lowest cost + cost of the losses it causes (losses must be produced). Look at `capacity` vs `cap_max_kW` (section 7.1).*
2. **What is the difference between the selected pipes in terms of cost, thermal losses and capacity?**
   *Hint: section 8 table – absolute and per-kW values.*
3. **Why might the optimizer favour one type near the beginning of the network (near producers) and another elsewhere?**
   *Hint: flows aggregate towards the producers; branches to single consumers only carry one peak.*
4. Report the design problem (section 6) and justify **all** 🔧 assumptions with units and sources.